# Initial Setup

In [18]:
crmp_url = "postgresql+psycopg2://crmp@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/climo-data-importer/.pgpass"
metnorth_url = "postgresql+psycopg2://metnorth_ro@/metnorth?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/climo-data-importer/.pgpass"

import logging
import pandas as pd
import sqlalchemy as sa
from sqlalchemy.orm import Session

crmp_engine = sa.create_engine(crmp_url, echo=False)
crmp_session = Session(crmp_engine)
metnorth_engine = sa.create_engine(metnorth_url, echo=False)
metnorth_session = Session(metnorth_engine)

## Pull networks from each session

In [20]:
q_network = sa.text("""
    SELECT * 
    FROM crmp.meta_network 
    WHERE network_name = :network_name
""")

with crmp_engine.begin() as connection:
    crmp_result = connection.execute(q_network, {"network_name": "EC_raw"})
    crmp_df = pd.DataFrame(crmp_result.fetchall(), columns=crmp_result.keys())

with metnorth_engine.begin() as connection:
    metnorth_result = connection.execute(q_network, {"network_name": "ECCC"})
    metnorth_df = pd.DataFrame(metnorth_result.fetchall(), columns=metnorth_result.keys())

display(crmp_df)
display(metnorth_df)


,network_id,network_name,description,virtual,publish,col_hex,contact_id,mod_time,mod_user,network_display_name
0,19,EC_raw,Environment and Climate Change Canada (raw obs...,None,True,#FF0000,None,2026-02-25 10:26:44.730224,tongli1997,EC_Raw


,network_id,network_name,description,virtual,publish,col_hex,contact_id,mod_time,mod_user,network_display_name
0,2,ECCC,Environment and Climate Change Canada,None,True,#ff0000,None,2025-09-17 12:02:06.131802,metnorth,ECCC


In [21]:
q_variables = sa.text("""
    SELECT *
    FROM crmp.meta_vars
    WHERE network_id = :network_id
""")

crmp_network_id = crmp_df["network_id"].iloc[0]
metnorth_network_id = metnorth_df["network_id"].iloc[0]

with crmp_engine.begin() as connection:
    crmp_result = connection.execute(q_variables, {"network_id": int(crmp_network_id)})
    crmp_df = pd.DataFrame(crmp_result.fetchall(), columns=crmp_result.keys())

with metnorth_engine.begin() as connection:
    metnorth_result = connection.execute(q_variables, {"network_id": int(metnorth_network_id)})
    metnorth_df = pd.DataFrame(metnorth_result.fetchall(), columns=metnorth_result.keys())

display(crmp_df)
display(metnorth_df)


,vars_id,network_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,mod_time,mod_user
0,542,19,Celsius,None,air_temperature,time: point,Hourly air temperature,air_temperature,Temperature (Point),air_temperature_point,2025-02-11 16:03:39.747374,crmp
1,543,19,km/h,None,wind_speed_of_gust,time: maximum,Maximum speed of wind gust,wind_gust_speed,Wind Gust (Max.),wind_speed_of_gust_maximum,2025-02-11 16:03:39.747374,crmp
2,545,19,Celsius,None,air_temperature,time: minimum,Minimum daily air temperature,air_temperature_yesterday_low,Temperature (Min.),air_temperature_minimum,2025-02-11 16:03:39.747374,crmp
3,544,19,Celsius,None,air_temperature,time: maximum,Maximum daily air temperature,air_temperature_yesterday_high,Temperature (Max.),air_temperature_maximum,2025-02-11 16:03:39.747374,crmp
4,551,19,Celsius,None,dew_point_temperature,time: point,Dew point,dew_point,Dew Point Temperature (Point),dew_point_temperature_point,2025-02-11 16:03:39.747374,crmp
5,550,19,km/h,None,wind_speed,time: point,Instantaneous wind speed,wind_speed,Wind Speed (Point),wind_speed_point,2025-02-11 16:03:39.747374,crmp
6,548,19,cm,None,thickness_of_snowfall_amount,time: sum,Daily snowfall,snow_amount,Snowfall Amount,thickness_of_snowfall_amount_sum,2025-02-11 16:03:39.747374,crmp
7,552,19,percent,None,relative_humidity,time: point,Relative humidity,relative_humidity,Relative Humidity (Point),relative_humidity_point,2025-02-11 16:03:39.747374,crmp
8,547,19,mm,None,thickness_of_rainfall_amount,time: sum,Daily rainfall amount,total_rain,Rainfall Amount,thickness_of_rainfall_amount_sum,2025-02-11 16:03:39.747374,crmp
9,613,19,celsius,None,air_temperature,t: maximum within days t: mean within months t...,Climatological mean of monthly mean maximum da...,Tx_Climatology,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean ...,2025-02-11 16:03:39.747374,crmp


,vars_id,net_var_name,unit,standard_name,cell_method,precision,long_description,display_name,short_name,network_id,mod_time,mod_user
0,25,Snow_Water_equivalent_precipitation,mm,lwe_thickness_of_snowfall_amount,time: point,None,water equivalent snowfall,Snow Fall Water Equivalent,None,2,2025-09-17 12:02:06.131802,metnorth
1,27,Humidity,%,relative_humidity,time: point,None,relative humidity in percent,Relative Humidity,None,2,2025-09-17 12:02:06.131802,metnorth
2,32,Pressure,millibar,air_pressure,time: point,None,Atmospheric pressure,Atmospheric Pressure,None,2,2025-09-17 12:02:06.131802,metnorth
3,31,Visibility,km,visibility_in_air,time: point,None,Visibility,Visibility,None,2,2025-09-17 12:02:06.131802,metnorth
4,33,Weather,NULL,NULL,NULL,None,NaN,Present Weather,None,2,2025-09-17 12:02:06.131802,metnorth
5,28,Wind_Speed,km h-1,wind_speed,time: point,None,wind speed,Wind Speed,None,2,2025-09-17 12:02:06.131802,metnorth
6,51,Spd_of_Max_Gust,km h-1,wind_speed_of_gust,time: maximum,None,Daily maximum wind speed,Max Gust Speed,None,2,2025-09-17 12:02:06.131802,metnorth
7,29,Wind_direction,degrees,wind_from_direction,time: point,None,wind direction,Wind Direction,None,2,2025-09-17 12:02:06.131802,metnorth
8,34,Lake_Evaporation,mm,surface_water_evaporation_amount,time: sum,None,Lake evaporation,Lake Evaporation Amount,None,2,2025-09-17 12:02:06.131802,metnorth
9,35,Pan_Evaporation,mm,surface_water_evaporation_amount,time: sum,None,Pan evaporation,Pan Evaporation Amount,None,2,2025-09-17 12:02:06.131802,metnorth


In [24]:
crmp_vars = crmp_df[["net_var_name", "vars_id", "display_name"]].copy()
crmp_vars["_key"] = crmp_vars["net_var_name"].str.lower()

metnorth_vars = metnorth_df[["net_var_name", "vars_id", "display_name"]].copy()
metnorth_vars["_key"] = metnorth_vars["net_var_name"].str.lower()

merged = pd.merge(
    crmp_vars.rename(columns={"net_var_name": "crmp_net_var_name", "vars_id": "crmp_vars_id", "display_name": "crmp_display_name"}),
    metnorth_vars.rename(columns={"net_var_name": "metnorth_net_var_name", "vars_id": "metnorth_vars_id", "display_name": "metnorth_display_name"}),
    on="_key",
    how="outer",
).rename(columns={"_key": "net_var_name"})[["net_var_name", "crmp_net_var_name", "metnorth_net_var_name", "crmp_vars_id", "metnorth_vars_id", "crmp_display_name", "metnorth_display_name"]]

# 0 = both matched, 1 = crmp only, 2 = metnorth only
has_crmp = merged["crmp_vars_id"].notna()
has_metnorth = merged["metnorth_vars_id"].notna()
merged["_order"] = 2  # metnorth only default
merged.loc[has_crmp & ~has_metnorth, "_order"] = 1   # crmp only
merged.loc[has_crmp & has_metnorth, "_order"] = 0    # both matched

merged = merged.sort_values("_order").drop(columns="_order").reset_index(drop=True)

display(merged)


,net_var_name,crmp_net_var_name,metnorth_net_var_name,crmp_vars_id,metnorth_vars_id,crmp_display_name,metnorth_display_name
0,wind_speed,wind_speed,Wind_Speed,550.0,28.0,Wind Speed (Point),Wind Speed
1,wind_direction,wind_direction,Wind_direction,553.0,29.0,Wind Direction (Mean),Wind Direction
2,total_precipitation,total_precipitation,Total_Precipitation,546.0,48.0,Precipitation Amount,Daily Precipitation
3,total_rain,total_rain,Total_Rain,547.0,46.0,Rainfall Amount,Daily Rainfall
4,air_temperature,air_temperature,NaN,542.0,NaN,Temperature (Point),NaN
5,air_temperature_yesterday_high,air_temperature_yesterday_high,NaN,544.0,NaN,Temperature (Max.),NaN
6,air_temperature_yesterday_low,air_temperature_yesterday_low,NaN,545.0,NaN,Temperature (Min.),NaN
7,dew_point,dew_point,NaN,551.0,NaN,Dew Point Temperature (Point),NaN
8,tendency_amount,tendency_amount,NaN,555.0,NaN,Air Pressure Tendency,NaN
9,snow_amount,snow_amount,NaN,548.0,NaN,Snowfall Amount,NaN
